# Tarea 5 · Sistemas de diálogo orientados a tareas (*task-oriented dialogue*)

*un asistente que entiende qué le piden y lo hace*

**PLE · el proyecto · cuaderno de arranque.** El proyecto es una tarea de
lenguaje, sobre **vuestros datos**, resuelta con **una herramienta clásica y
una moderna**, medidas igual contra **un oro** (las respuestas correctas,
hechas a mano por personas), y los fallos de cada una explicados. Este
cuaderno hace todo eso con unos pocos casos de ejemplo. Vuestro trabajo
empieza cambiándolos por los vuestros: la última sección dice cómo, paso a
paso.

| | |
|---|---|
| **Lo que entra** | lo que dice el usuario (aquí, a un asistente de reservas de pistas de pádel) |
| **Lo que sale** | **la acción** del asistente: reservar (con pista, día y hora), consultar, anular, preguntar lo que falta, o decir que eso no lo hace |
| **La clásica** | **reglas** que buscan la **intención** (qué quiere) y rellenan los **huecos** (*slots*: día, hora, pista) |
| **La moderna** | un **LLM** con instrucciones, que devuelve la acción en JSON |
| **La medida** | el % de peticiones con la acción y los huecos del oro; en el proyecto, conversaciones enteras: tareas completadas y turnos |
| **En clase** | la sesión de conversar, más adelante; las expresiones regulares, en la de procesamiento de texto (miércoles 9 de septiembre) |
| **La ficha** | `ficha-5-dialogo.pdf`, en este mismo zip: lo que hace falta saber de la tarea hasta su sesión |

## 0. Preparación, en Colab

Este cuaderno se ejecuta en **Google Colab** (colab.research.google.com), en
CPU: no hace falta cambiar el tipo de entorno.

1. Descomprimid en vuestro ordenador el zip de esta tarea, `tarea-5-dialogo.zip`
   (del Campus): trae este cuaderno y la carpeta `datos/`.
2. En Colab: **Archivo → Subir cuaderno** → `tarea-5-dialogo.ipynb`.
3. En Colab, el icono de la **carpeta**, a la izquierda → **Subir** → el zip
   entero, `tarea-5-dialogo.zip`. Esperad a que aparezca en la lista.
4. Ejecutad la celda de abajo: instala lo que falte, descomprime el zip y
   dice si está cada fichero que el cuaderno lee. Fuera de Colab (Jupyter, VS
   Code) funciona igual: la celda instala lo que vuestro Python no tenga.

Si Colab se desconecta, borra los ficheros: volved a subir el zip y a
ejecutar esta celda. **La primera vez se descargan los modelos**: el LLM, Qwen (un giga).
**El aviso de Hugging Face sobre el token** (`HF_TOKEN`, «The secret HF_TOKEN does
not exist…»): **ignoradlo**. Ningún cuaderno de arranque lo necesita: los modelos
se descargan igual sin él.

**No hace falta GPU**: todo corre en la CPU del Colab gratuito. Las celdas que
tardan más de un minuto lo dicen al empezar.

In [ ]:
import glob
import importlib.util
import os
import shutil
import subprocess
import sys
import zipfile
import json
import time
import re

EN_COLAB = 'COLAB_RELEASE_TAG' in os.environ          # ¿estamos en Google Colab?
CARPETA_ENV = "env_rasa"

# Determinar rutas de ejecutables del entorno para Rasa según el sistema operativo
if os.name == 'nt':  # Windows
    python_rasa = os.path.join(CARPETA_ENV, "Scripts", "python.exe")
    rasa_bin = os.path.join(CARPETA_ENV, "Scripts", "rasa.exe")
else:  # Linux / Google Colab
    python_rasa = os.path.join(CARPETA_ENV, "bin", "python")
    rasa_bin = os.path.join(CARPETA_ENV, "bin", "rasa")

# Si el entorno de python no existe en el equipo
if not os.path.exists(rasa_bin):
    print("No se encontró el entorno para Rasa, configurando un nuevo entorno aislado...")
    
    if EN_COLAB:
        # En Colab (Ubuntu), instalamos Python 3.10 y venv desde apt para evitar conflictos con Python 3.11/3.12
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "python3.10", "python3.10-venv"], check=True)
        subprocess.run(["python3.10", "-m", "venv", CARPETA_ENV], check=True)
    elif os.name == 'nt':
        # En Windows apunta a la instalación de Python 3.10 que debe existir
        subprocess.run(["py", "-3.10", "-m", "venv", CARPETA_ENV], check=True)
    else:
        # En Linux/macOS apunta a la instalación de Python 3.10 que debe existir
        subprocess.run(["python3.10", "-m", "venv", CARPETA_ENV], check=True)

    print("Instalando Rasa (esto tardará un par de minutos la primera vez)...")
    
    # Con -q no imprimimos los logs en la terminal, tambien especificamos la versión de Rasa por si cambia en un futuro además añadimos otros 
    # atributos opcionales por precaución como la librería requests y las herramientas internas de empaquetado de python setuptools y wheel
    subprocess.run([python_rasa, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
    subprocess.run([python_rasa, "-m", "pip", "install", "-q", "rasa==3.6.15", "requests"], check=True)

    print("Rasa instalado correctamente en: ", CARPETA_ENV)
else:
    print("Entorno de Rasa detectado y listo.")

# Instalar lo que falte, en Colab o en vuestro ordenador (módulo → paquete de pip)
PAQUETES = {'nltk': 'nltk', 'torch': 'torch', 'transformers': 'transformers'}
faltan = [paquete for modulo, paquete in PAQUETES.items()
          if importlib.util.find_spec(modulo) is None]
if faltan:
    print('Instalando:', ' '.join(faltan))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltan], check=True)
    importlib.invalidate_caches()

# Descomprimir el zip de la tarea, se llame como se llame
# (el navegador a veces le añade « (1)» al nombre).
if not os.path.exists('datos'):
    for zip_subido in glob.glob('*.zip'):
        zipfile.ZipFile(zip_subido).extractall('.')
        print('Descomprimido:', zip_subido)

print('Carpeta de trabajo:', os.getcwd())
FICHEROS_QUE_LEE = ['datos/casos.tsv']
for fichero in FICHEROS_QUE_LEE:
    print('  está  ' if os.path.exists(fichero) else '  FALTA ', fichero)

Instalando: nltk torch transformers
Carpeta de trabajo: c:\Users\telmo\OneDrive\Escritorio\UCM TKB\Curso 4\PLE\Proyecto Final\proyecto-asistente-git
  está   datos/casos.tsv


## 1. Los casos: `datos/casos.tsv`

Diez peticiones de un solo turno, inventadas, cada una con **la acción
correcta** (el oro). Columnas: `peticion`, `accion`, y los huecos que esa
acción lleva: `pista`, `dia` y `hora` para reservar; `dia` y `hora` para
consultar; `dia` para anular; `falta` (qué dato pedir) para preguntar. Los
huecos que la acción no lleva van vacíos.

`leer_tsv(ruta)` lee un fichero **separado por tabuladores** (`.tsv`, se abre
con cualquier hoja de cálculo): la primera línea dice las columnas, y cada
línea siguiente es un caso. Devuelve una lista con un diccionario por línea,
`{columna: valor}`; los valores llegan como texto.

`accion_del_oro(fila)` se queda con la acción y los huecos que no están
vacíos: un diccionario como `{'accion': 'reservar', 'pista': '2', 'dia':
'jueves', 'hora': '18'}`.

In [ ]:
CARPETA_PROYECTO = "proyecto-rasa"

print("Entrenando modelo de Rasa...")

inicio_train = time.time()
subprocess.run([rasa_bin, "train"], cwd=CARPETA_PROYECTO, check=True)

print(f"Entrenamiento completado en {time.time() - inicio_train:.2f} segundos.")

5 peticiones.  La primera:
    Crear un  uevo repositiorio local -> {'accion': 'crear_repositiorio', 'directorio': '.'}


## 2. La medida: acción y huecos

La **acción** es la **intención** (*intent*) de la persona: reservar,
preguntar, anular. Los **huecos** (*slots*) son los datos que la tarea
necesita, cada uno con su **valor**. Una respuesta **acierta** si la acción es
la del oro **y cada hueco del oro tiene su valor** (se compara como texto: `18`
y `'18'` valen lo mismo). Esto mide **turno a turno**; en conversaciones
enteras se mide la **tasa de tareas completadas** (la sesión de sistemas de
diálogo).

- `acierta(accion, correcta)`: ¿es ésta la acción correcta?
- `porcentaje_de_aciertos(acciones, oro)`: el % de peticiones acertadas.

In [ ]:
def acierta(accion: dict, correcta: dict) -> bool:
    """La acción y todos los huecos del oro, iguales (comparados como texto)."""
    for clave, valor in correcta.items():
        if str(accion.get(clave)) != str(valor):
            return False
    return True


def porcentaje_de_aciertos(acciones: list, oro: list) -> float:
    """% de peticiones en que la acción y sus huecos son los del oro."""
    aciertos = sum(1 for accion, correcta in zip(acciones, oro) if acierta(accion, correcta))
    return 100 * aciertos / len(oro)

## 3. La clásica: reglas que rellenan huecos

`con_reglas(peticion, dias)` hace dos cosas, como los asistentes de antes de
los LLM:
1. **rellena los huecos** con expresiones regulares: un día de la lista
   `DIAS`, «a las N» o «las N» para la hora, «pista N» para la pista;
2. **decide la intención** por palabras clave: *anul-*/*cancel-* es anular;
   *libre*/*hay alguna* es consultar; *reserv-*/*quiero*/*jugar*/*pista* es
   reservar, y si falta el día o la hora, pregunta por él; lo demás, fuera.

In [ ]:
import requests

# Arrancar servidores de Rasa en segundo plano apuntando al entorno aislado
proc_actions = subprocess.Popen(
    [rasa_bin, "run", "actions"],
    cwd=CARPETA_PROYECTO,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

proc_rasa = subprocess.Popen(
    [rasa_bin, "run", "--enable-api"],
    cwd=CARPETA_PROYECTO,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Esperar a que la API local responda
print("⏳ Iniciando motores de Rasa...")

while True:
    try:
        if requests.get("http://localhost:5005/").status_code == 200:
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
        
print("🚀 Servidores de Rasa activos.")

def preguntar_rasa(texto: str) -> str:
    """Devuelve la respuesta de Rasa en formato texto plano, idéntico a preguntar() de Qwen."""
    url = "http://localhost:5005/webhooks/rest/webhook"
    res = requests.post(url, json={"sender": "eval_user", "message": texto}).json()
    return res[0]["text"] if res else "SIN_RESPUESTA"

# --- PRUEBA COMPARATIVA EN VIVO ---
consulta = "crea la rama prueba-clase y muéveme ahí"

t0 = time.time()
resp_rasa = preguntar_rasa(consulta)
t_rasa = time.time() - t0

t0 = time.time()
resp_qwen = preguntar([
    {"role": "system", "content": "Devuelve solo el comando git exacto sin explicaciones."},
    {"role": "user", "content": consulta}
], tokens_nuevos=30)
t_qwen = time.time() - t0

print(f"Consulta: {consulta}\n")
print(f"Rasa ({t_rasa*1000:.0f} ms): {resp_rasa}")
print(f"Qwen ({t_qwen*1000:.0f} ms): {resp_qwen}")


las reglas: 0 % de aciertos


## 4. La moderna: un LLM con instrucciones

**La herramienta moderna: un LLM.** `Qwen2.5-0.5B-Instruct` es un modelo de
chat pequeño (unos 500 millones de pesos; se descarga una vez, un giga) que
corre en Colab sin GPU. Es pequeño a propósito: es lo que podéis ejecutar
vosotros mismos, y sus fallos se ven. **También podéis medir el chat que
uséis** (Gemini, con la cuenta de la UCM, o el que sea): la sección 4b dice
dónde pegar sus respuestas.

La celda de abajo:
1. descarga el modelo y su tokenizador (el que parte el texto en tokens);
2. lo carga en `float32`: el modelo se guardó en `bfloat16`, un formato de
   números que las CPU de Colab hacen muy despacio;
3. define `preguntar(mensajes, tokens_nuevos)`: le da al LLM una
   conversación (una lista de mensajes, cada uno con su papel, `system` o
   `user`, y su texto) y devuelve su respuesta. Elige siempre el token más
   probable (**descodificación voraz**, *greedy*), así que la misma pregunta
   da siempre la misma respuesta; `tokens_nuevos` es lo más larga que puede
   ser.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()          # que sólo hable si algo falla

NOMBRE_DEL_LLM = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizador = AutoTokenizer.from_pretrained(NOMBRE_DEL_LLM)
llm = AutoModelForCausalLM.from_pretrained(NOMBRE_DEL_LLM, dtype=torch.float32)
llm.eval()


def preguntar(mensajes: list, tokens_nuevos: int) -> str:
    """La respuesta del LLM a una conversación, eligiendo siempre el token más probable."""
    entrada = tokenizador.apply_chat_template(mensajes, add_generation_prompt=True,
                                              return_tensors='pt', return_dict=True)
    with torch.no_grad():
        salida = llm.generate(**entrada, max_new_tokens=tokens_nuevos, do_sample=False)
    largo_de_la_pregunta = entrada['input_ids'].shape[1]
    tokens_de_la_respuesta = salida[0][largo_de_la_pregunta:]
    return tokenizador.decode(tokens_de_la_respuesta, skip_special_tokens=True).strip()

C:\Users\telmo\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\telmo\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\telmo\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In o

`INSTRUCCIONES` va como mensaje de **sistema** (`system`): lo que el asistente
es y el formato de cada acción, en JSON. `con_el_llm(peticion, instrucciones,
tokens_nuevos)` busca en la respuesta el primer `{…}` y lo lee como JSON con
`accion_del_texto`; si no hay o no se puede leer, devuelve `{'accion': 'no es
JSON'}`, que cuenta como fallo.

**La celda siguiente tarda hasta un minuto** en el Colab gratuito, y no escribe nada hasta que acaba: no está colgada, esperad.

In [ ]:
print('Esta celda tarda hasta un minuto en el Colab gratuito, y no escribe nada hasta que acaba: esperad.', flush=True)

INSTRUCCIONES = ('Eres el asistente de reservas de pistas de pádel de la universidad (pistas 1, 2 y 3; '
                 'de 9 a 21 h). Para cada mensaje, responde SÓLO un JSON con la acción: '
                 '{"accion": "reservar", "pista": n, "dia": "...", "hora": h} si están los tres datos; '
                 '{"accion": "consultar", "dia": "...", "hora": h} si pregunta si hay sitio; '
                 '{"accion": "anular", "dia": "..."}; '
                 '{"accion": "preguntar", "falta": "dia" u "hora"} si falta un dato para reservar; '
                 '{"accion": "fuera"} si pide otra cosa.')
TOKENS_NUEVOS = 60


def accion_del_texto(texto: str) -> dict:
    """El primer {...} del texto, leído como JSON; {'accion': 'no es JSON'} si no se puede."""
    encontrado = re.search(r'\{.*?\}', texto, re.S)
    if not encontrado:
        return {'accion': 'no es JSON'}
    try:
        return json.loads(encontrado.group(0))
    except ValueError:
        return {'accion': 'no es JSON'}


def con_el_llm(peticion: str, instrucciones: str, tokens_nuevos: int) -> dict:
    """La acción que devuelve el LLM para una petición."""
    respuesta = preguntar([{'role': 'system', 'content': instrucciones},
                           {'role': 'user', 'content': peticion}], tokens_nuevos)
    return accion_del_texto(respuesta)


del_llm = [con_el_llm(peticion, INSTRUCCIONES, TOKENS_NUEVOS) for peticion in PETICIONES]
aciertos_llm = porcentaje_de_aciertos(del_llm, ORO)
print(f'el LLM: {aciertos_llm:.0f} % de aciertos')

Esta celda tarda hasta un minuto en el Colab gratuito, y no escribe nada hasta que acaba: esperad.
el LLM: 0 % de aciertos


### 4b. El chat que uséis (opcional)

Si queréis medir también un chat grande (Gemini, ChatGPT…), hacedle la misma
petición que al LLM de arriba, **caso por caso y en el mismo orden**, y pegad
cada respuesta, entre comillas triples (`"""…"""`), en la lista
`RESPUESTAS_DEL_CHAT`. Dadle las mismas instrucciones (`INSTRUCCIONES`) antes de la primera petición. Si la lista está vacía, la celda sólo lo dice.

In [ ]:
RESPUESTAS_DEL_CHAT = [
    # """{"accion": "reservar", "pista": 2, "dia": "jueves", "hora": 18}""",
]

if RESPUESTAS_DEL_CHAT:
    del_chat = [accion_del_texto(respuesta) for respuesta in RESPUESTAS_DEL_CHAT]
    aciertos_chat = porcentaje_de_aciertos(del_chat, ORO)
    print(f'el chat: {aciertos_chat:.0f} % de aciertos')
else:
    print('Sin respuestas pegadas: se mide sólo el LLM de arriba.')

Sin respuestas pegadas: se mide sólo el LLM de arriba.


## 5. Las dos, medidas igual, y sus fallos

**Cuidado con el número de las reglas**: se escribieron mirando estos diez
casos, así que es un número de *dev* (desarrollo), como el de la clase más
frecuente en la sesión de etiquetar II. Con peticiones nuevas, escritas por
otras personas (el *test*, la prueba), bajará: eso es lo que mide el proyecto.
Debajo, **cada petición con algún fallo**: el oro y lo que hizo cada una.

In [ ]:
# Conjunto de datos de prueba (Frase del usuario, Comando exacto esperado)
dataset_test = [
    ("crea la rama login-fix y muéveme ahí", "git checkout -b login-fix"),
    ("guarda todo lo que hice con el mensaje 'ajuste visual'", 'git add . && git commit -m "ajuste visual"'),
    ("iniciar nuevo repositorio local", "git init"),
    ("clonar proyecto de https://github.com", "git clone https://github.com")
]

aciertos_rasa = 0
aciertos_qwen = 0

print(f"{'CONSULTA':<45} | {'ESPERADO':<35} | {'RASA':<30} | {'QWEN':<30}")
print("-" * 150)

for frase, esperado in dataset_test:
    # 1. Consultar a Rasa
    res_rasa = preguntar_rasa(frase).strip()
    if res_rasa == esperado:
        aciertos_rasa += 1

    # 2. Consultar a Qwen
    mensajes = [
        {"role": "system", "content": "Eres un asistente de Git. Responde únicamente con el comando exacto, sin markdown ni explicaciones."},
        {"role": "user", "content": frase}
    ]
    res_qwen = preguntar(mensajes, tokens_nuevos=40).strip()
    if res_qwen == esperado:
        aciertos_qwen += 1
        
    print(f"{frase[:43]:<45} | {esperado[:33]:<35} | {res_rasa[:28]:<30} | {res_qwen[:28]:<30}")

total = len(dataset_test)
print("-" * 150)
print(f"Precisión Exacta (Exact Match) Rasa: {(aciertos_rasa / total) * 100:.1f}% ({aciertos_rasa}/{total})")
print(f"Precisión Exacta (Exact Match) Qwen: {(aciertos_qwen / total) * 100:.1f}% ({aciertos_qwen}/{total})")



# ---------- old -----------
"""
print(f'{"":10}{"aciertos":>10}')
print(f'{"reglas":10}{aciertos_reglas:>9.0f}%')
print(f'{"el LLM":10}{aciertos_llm:>9.0f}%')
for peticion, correcta, de_uno, del_otro in zip(PETICIONES, ORO, de_reglas, del_llm):
    if not acierta(de_uno, correcta) or not acierta(del_otro, correcta):
        print()
        print(peticion)
        print(f'    oro:    {correcta}')
        print(f'    reglas: {de_uno}')
        print(f'    el LLM: {del_otro}')
"""

## 6. El segundo anotador: cuánto se fía uno del oro

El oro lo hacen personas, y dos personas no siempre coinciden. **El acuerdo
entre anotadores** es el porcentaje de casos en que dos personas, cada una
por su cuenta y sin mirar lo de la otra, dan la misma respuesta. Es **el
techo**: ninguna herramienta puede medirse por encima de lo que las personas
coinciden entre sí (en la sesión de etiquetar, lunes 19 de octubre, el
acuerdo de AnCora con nuestro oro era del 98,4 %).

**Cómo se hace en el proyecto**, también si trabajáis solos:
1. Otra persona hace el oro de **veinte casos** por su cuenta, con las mismas
   instrucciones escritas que seguisteis vosotros: vuestra pareja; alguien del
   curso (en la cabecera de la propuesta decís si necesitáis segundo anotador
   y si podéis anotar veinte casos para otra persona, y os emparejamos); o alguien de fuera del curso que conozca la
   lengua o el tema.
2. Se calcula el acuerdo en esos veinte, con la función de abajo.
3. Donde no coincidís, se habla y se decide; si las instrucciones eran
   ambiguas, se aclaran por escrito. El número y los desacuerdos van en el
   informe.

**Aquí, cada caso es una petición**, y lo que se anota a mano es **la acción
correcta** (el oro de §1). Dos personas lo deciden por separado para las
mismas peticiones, y se compara la acción. Abajo, **un ejemplo inventado** con
seis peticiones. En una conversación abierta, lo mismo con las puntuaciones
de la rúbrica.

In [ ]:
def acuerdo(respuestas_1: list, respuestas_2: list) -> float:
    """% de casos en que dos anotadores dan la misma respuesta (las dos listas, en el mismo orden)."""
    iguales = sum(1 for una, otra in zip(respuestas_1, respuestas_2) if una == otra)
    return 100 * iguales / len(respuestas_1)


acciones_anotador_1 = ['reservar', 'consultar', 'preguntar', 'anular', 'fuera', 'reservar']
acciones_anotador_2 = ['reservar', 'consultar', 'reservar', 'anular', 'fuera', 'reservar']   # inventado
acuerdo_del_ejemplo = acuerdo(acciones_anotador_1, acciones_anotador_2)
print(f'{len(acciones_anotador_1)} peticiones; acuerdo {acuerdo_del_ejemplo:.1f} %')

6 peticiones; acuerdo 83.3 %


## 7. Los mandos: qué cambiar, y qué esperáis que pase

Cada parámetro que cambia el resultado está declarado en una celda, con su
valor. Para cada uno: **escribid antes qué esperáis**, cambiadlo y volved a
ejecutar desde su celda hasta el final.

| Dónde | Cambiad | Antes de ejecutar, pensad |
|---|---|---|
| `datos/casos.tsv` | añadid cinco peticiones nuevas, escritas por otra persona que no haya visto las reglas | ¿cuánto baja el % de las reglas? ¿y el del LLM? |
| `INSTRUCCIONES` | quitad los ejemplos de JSON y dejad sólo la lista de acciones | ¿cuántas respuestas dejan de ser JSON? |
| `NOMBRE_DEL_LLM` | 'Qwen/Qwen2.5-1.5B-Instruct' | ¿sube el % del LLM? |

---

## 8. Vuestro proyecto, paso a paso

1. **Ejecutad este cuaderno tal cual** (**Entorno de ejecución → Ejecutar
   todo**) y mirad los fallos de §5.
2. **Cambiad los casos por los vuestros.** Abrid `datos/casos.tsv` con una
   hoja de cálculo, borrad los de ejemplo y escribid **entre 5 y 10 casos
   vuestros**, cada uno con su acción correcta (o, en conversación abierta, con lo que una buena respuesta tiene que hacer), con las mismas columnas. Guardadlo como texto
   separado por tabuladores, con el mismo nombre, subidlo a Colab y ejecutad
   todo otra vez.
3. **Para la propuesta revisada** (una página, sola o en pareja; se entrega en el
   Campus el **martes 17 de noviembre, a las 20:00**): la tabla de §5 con
   vuestros casos, y **dos fallos de cada herramienta**, copiados tal cual,
   con lo que creéis que pasa.
4. **El oro completo**, hecho a mano: entre 50 y 100 peticiones, o entre 20 y 30 conversaciones, con las
   instrucciones escritas de cómo se decide cada respuesta. Veinte de ellos,
   también por una segunda persona (§6).
5. **Las dos herramientas, medidas igual** sobre todo el oro, y **los fallos
   clasificados**: no basta el número; hay que decir por qué falla cada una
   y dar ejemplos.
6. **¿Es real la diferencia?** Con pocos casos, dos números parecidos pueden
   ser suerte: el **remuestreo** (*bootstrap*) lo decide (la sesión de
   sistemas de diálogo lo explica paso a paso).

**Ideas para los datos**: un servicio estrecho (reservas, un juego, comandos de una herramienta, una biblioteca); peticiones escritas por personas que no vieron vuestras reglas; conversaciones enteras, con tareas completadas y turnos; o respuestas a mensajes con emociones, con una rúbrica a ciegas.

Las dudas, por mensaje al profesor en el Campus. El cuaderno es vuestro:
cambiad lo que haga falta.